In [3]:
import polars as pl

df = pl.read_parquet("data/top_pairs/daily_top_pairs_573_90.parquet")
print(df.schema)
print(df.head(10))

Schema({'date': Date, 'leader': String, 'follower': String, 'l_hat': Float64, 'sigma_l': Float64, 'cost': Float64})
shape: (10, 6)
┌────────────┬─────────┬──────────┬───────────┬──────────┬──────────┐
│ date       ┆ leader  ┆ follower ┆ l_hat     ┆ sigma_l  ┆ cost     │
│ ---        ┆ ---     ┆ ---      ┆ ---       ┆ ---      ┆ ---      │
│ date       ┆ str     ┆ str      ┆ f64       ┆ f64      ┆ f64      │
╞════════════╪═════════╪══════════╪═══════════╪══════════╪══════════╡
│ 2015-01-02 ┆ MRK.N   ┆ LOW.N    ┆ 1.050847  ┆ 1.850778 ┆ 0.126516 │
│ 2015-01-02 ┆ MMM.N   ┆ GD.N     ┆ -1.050328 ┆ 1.936544 ┆ 0.099482 │
│ 2015-01-02 ┆ BLK.N   ┆ CMCSA.OQ ┆ -1.039419 ┆ 1.976129 ┆ 0.140185 │
│ 2015-01-02 ┆ ALL.N   ┆ SO.N     ┆ 1.168831  ┆ 2.040091 ┆ 0.134003 │
│ 2015-01-02 ┆ BIIB.OQ ┆ LOW.N    ┆ 1.076923  ┆ 2.061504 ┆ 0.178299 │
│ 2015-01-02 ┆ WFC.N   ┆ GM.N     ┆ -1.337553 ┆ 2.079634 ┆ 0.143226 │
│ 2015-01-02 ┆ FOXA.OQ ┆ PCLN.OQ  ┆ -1.056842 ┆ 2.118324 ┆ 0.113698 │
│ 2015-01-02 ┆ CAT.N   ┆ KO.N

In [5]:
print(df.select([
    pl.col("date").min().alias("min_date"),
    pl.col("date").max().alias("max_date"),
    pl.n_unique("date").alias("n_days"),
    pl.len().alias("n_rows"),
]))

print(df.select([
    pl.col("leader").n_unique().alias("n_leaders"),
    pl.col("follower").n_unique().alias("n_followers"),
]))

shape: (1, 4)
┌────────────┬────────────┬────────┬────────┐
│ min_date   ┆ max_date   ┆ n_days ┆ n_rows │
│ ---        ┆ ---        ┆ ---    ┆ ---    │
│ date       ┆ date       ┆ u32    ┆ u32    │
╞════════════╪════════════╪════════╪════════╡
│ 2015-01-02 ┆ 2017-04-12 ┆ 573    ┆ 5730   │
└────────────┴────────────┴────────┴────────┘
shape: (1, 2)
┌───────────┬─────────────┐
│ n_leaders ┆ n_followers │
│ ---       ┆ ---         │
│ u32       ┆ u32         │
╞═══════════╪═════════════╡
│ 90        ┆ 90          │
└───────────┴─────────────┘


In [7]:
print(df.group_by("date").len().sort("len", descending=True).head(10))

shape: (10, 2)
┌────────────┬─────┐
│ date       ┆ len │
│ ---        ┆ --- │
│ date       ┆ u32 │
╞════════════╪═════╡
│ 2016-08-29 ┆ 10  │
│ 2016-09-28 ┆ 10  │
│ 2016-05-23 ┆ 10  │
│ 2015-04-17 ┆ 10  │
│ 2015-12-14 ┆ 10  │
│ 2016-01-19 ┆ 10  │
│ 2017-02-09 ┆ 10  │
│ 2015-04-23 ┆ 10  │
│ 2016-05-26 ┆ 10  │
│ 2015-08-17 ┆ 10  │
└────────────┴─────┘


In [17]:
pairs = pl.read_parquet("data/top_pairs/daily_top_pairs_573_90.parquet")
print(pairs.select([
    pl.col("l_hat").abs().mean().alias("mean_abs_lag"),
    pl.col("l_hat").abs().max().alias("max_abs_lag"),
]))

shape: (1, 2)
┌──────────────┬─────────────┐
│ mean_abs_lag ┆ max_abs_lag │
│ ---          ┆ ---         │
│ f64          ┆ f64         │
╞══════════════╪═════════════╡
│ 2.348955     ┆ 9.204513    │
└──────────────┴─────────────┘


In [9]:
import polars as pl

df = pl.read_parquet("data/selected/SP100/bbo/AAPL.OQ.parquet")
print(df.schema)
print(df.head(5))


Schema({'timestamp': Datetime(time_unit='us', time_zone='America/New_York'), 'mid_price_return': Float64})
shape: (5, 2)
┌────────────────────────────────┬──────────────────┐
│ timestamp                      ┆ mid_price_return │
│ ---                            ┆ ---              │
│ datetime[μs, America/New_York] ┆ f64              │
╞════════════════════════════════╪══════════════════╡
│ 2015-01-02 09:32:00 EST        ┆ 0.000404         │
│ 2015-01-02 09:33:00 EST        ┆ -0.000045        │
│ 2015-01-02 09:34:00 EST        ┆ -0.001033        │
│ 2015-01-02 09:35:00 EST        ┆ -0.001124        │
│ 2015-01-02 09:36:00 EST        ┆ 0.001531         │
└────────────────────────────────┴──────────────────┘


In [15]:
df.select([
    pl.col("timestamp").min().alias("min_ts"),
    pl.col("timestamp").max().alias("max_ts"),
    pl.len().alias("n_rows"),
    pl.col("timestamp").dt.date().n_unique().alias("n_days"),
])

min_ts,max_ts,n_rows,n_days
"datetime[μs, America/New_York]","datetime[μs, America/New_York]",u32,u32
2015-01-02 09:32:00 EST,2017-03-31 16:00:00 EDT,219785,565


In [13]:
print(
    df.with_columns(pl.col("timestamp").dt.date().alias("date"))
      .group_by("date")
      .len()
      .sort("len", descending=True)
      .head(5)
)

shape: (5, 2)
┌────────────┬─────┐
│ date       ┆ len │
│ ---        ┆ --- │
│ date       ┆ u32 │
╞════════════╪═════╡
│ 2015-08-20 ┆ 389 │
│ 2015-04-17 ┆ 389 │
│ 2016-01-07 ┆ 389 │
│ 2016-09-16 ┆ 389 │
│ 2017-02-06 ┆ 389 │
└────────────┴─────┘


## Trading

## Trading, chiller condition

In [39]:
from __future__ import annotations

import math
from pathlib import Path
from dataclasses import dataclass
import numpy as np
import polars as pl

RETURNS_DIR = Path("data/selected/SP100/bbo")
PAIRS_PATH  = Path("data/top_pairs/daily_top_pairs_573_90.parquet")

def load_ticker_day(ticker: str, day: pl.Date) -> pl.DataFrame:
    path = RETURNS_DIR / f"{ticker}.parquet"
    return (
        pl.scan_parquet(path)
        .filter(pl.col("timestamp").dt.date() == day)
        .select(
            pl.col("timestamp"),
            pl.col("mid_price_return").alias("r"),
        )
        .sort("timestamp")
        .collect()
    )

'''
def rolling_mean_std(x: np.ndarray, window: int) -> tuple[np.ndarray, np.ndarray]:
    n = x.size
    mean = np.full(n, np.nan, dtype=np.float64)
    std  = np.full(n, np.nan, dtype=np.float64)

    if window <= 1 or n < window:
        return mean, std

    csum = np.cumsum(np.insert(x, 0, 0.0))
    csum2 = np.cumsum(np.insert(x * x, 0, 0.0))

    for t in range(window - 1, n):
        s  = csum[t + 1] - csum[t + 1 - window]
        s2 = csum2[t + 1] - csum2[t + 1 - window]
        m = s / window
        v = s2 / window - m * m
        if v < 0:
            v = 0.0
        mean[t] = m
        std[t] = math.sqrt(v)

    return mean, std
'''

def rolling_mean_std_nan_safe(x: np.ndarray, window: int) -> tuple[np.ndarray, np.ndarray]:
    """
    Rolling mean/std over a fixed window, but NaN-safe.
    If a window has < window valid points, returns NaN for that t (strict).
    Returns arrays same length as x, NaN until enough valid history.
    """
    n = x.size
    mean = np.full(n, np.nan, dtype=np.float64)
    std  = np.full(n, np.nan, dtype=np.float64)

    if window <= 1 or n < window:
        return mean, std

    # valid mask + replace NaNs by 0 for sums
    valid = np.isfinite(x)
    x0 = np.where(valid, x, 0.0)

    # cumulative sums of values and squares
    csum  = np.cumsum(np.insert(x0, 0, 0.0))
    csum2 = np.cumsum(np.insert(x0 * x0, 0, 0.0))

    # cumulative counts of valid samples
    ccount = np.cumsum(np.insert(valid.astype(np.int32), 0, 0))

    for t in range(window - 1, n):
        s   = csum[t + 1]  - csum[t + 1 - window]
        s2  = csum2[t + 1] - csum2[t + 1 - window]
        cnt = ccount[t + 1] - ccount[t + 1 - window]

        # strict: require full window of valid points
        if cnt < window:
            continue

        m = s / window
        v = s2 / window - m * m
        if v < 0:
            v = 0.0

        mean[t] = m
        std[t]  = math.sqrt(v)

    return mean, std



@dataclass
class PairTradeParams:
    lookback: int = 20      # like your other code's d
    k: float = 2.0          # Bollinger width
    z_exit: float = 0.0     # use mean-cross exit (0 means exit at mean)
    tc_bps: float = 0.0
    max_lag_minutes: int = 30
    enter_on_next_bar: bool = True

def trade_one_pair_one_day_market_neutral(
    r_leader: np.ndarray,
    r_follower: np.ndarray,
    lag: int,
    params: PairTradeParams,
) -> tuple[float, int, int]:
    """
    Market-neutral:
      pos = +1  => long follower, short leader
      pos = -1  => short follower, long leader

    Trigger:
      Build spread m_t = P_L(t-lag) - P_F(t)
      Bollinger on m_t
      if m > upper => pos = -1
      if m < lower => pos = +1
      exit when m crosses mean (or within band if you want)
    Returns: (pnl, entries, exits)
    """
    n = min(len(r_leader), len(r_follower))
    if n == 0:
        return 0.0, 0, 0

    lag = int(lag)
    if lag <= 0 or lag > params.max_lag_minutes:
        return 0.0, 0, 0

    rL = np.where(np.isfinite(r_leader[:n]), r_leader[:n], 0.0)
    rF = np.where(np.isfinite(r_follower[:n]), r_follower[:n], 0.0)

    # "price proxies"
    pL = np.cumsum(rL)
    pF = np.cumsum(rF)

    # lagged leader price proxy
    pred = np.full(n, np.nan, dtype=np.float64)
    pred[lag:] = pL[:-lag]

    # spread / mispricing
    m = pred - pF

    mu, sig = rolling_mean_std_nan_safe(m, params.lookback)
    #assert not np.all(np.isnan(mu)), "mu is ALL NaN — rolling stats are broken"
    # avoid nonsense early
    sig[sig < 1e-12] = np.nan

    upper = mu + params.k * sig
    lower = mu - params.k * sig

    # trading state
    pos = 0.0
    pnl = 0.0
    entries = 0
    exits = 0
    tc = params.tc_bps / 10000.0

    start = max(params.lookback - 1, lag)

    # optional: trade with 1-bar delay to avoid same-bar execution
    pending_pos: float | None = None

    for t in range(start, n):
        # apply pending position change at bar open (one-bar delay)
        if params.enter_on_next_bar and pending_pos is not None:
            if pending_pos != pos:
                # costs on change (two legs)
                pnl -= tc * (abs(pending_pos - pos) * 2.0)
                # count entry/exit
                if pos == 0.0 and pending_pos != 0.0:
                    entries += 1
                if pos != 0.0 and pending_pos == 0.0:
                    exits += 1
                pos = pending_pos
            pending_pos = None

        # accrue pnl for this minute (market-neutral)
        pnl += pos * rF[t] - pos * rL[t]

        mt = m[t]
        if not np.isfinite(mt) or not np.isfinite(mu[t]) or not np.isfinite(upper[t]) or not np.isfinite(lower[t]):
            continue

        # exit at mean-cross (clean + common)
        if pos != 0.0:
            if (pos == +1.0 and mt >= mu[t] + params.z_exit) or (pos == -1.0 and mt <= mu[t] - params.z_exit):
                # close
                desired = 0.0
                if params.enter_on_next_bar:
                    pending_pos = desired
                else:
                    pnl -= tc * (abs(desired - pos) * 2.0)
                    exits += 1
                    pos = desired
                continue

        # entry / flip
        desired = pos
        if mt > upper[t]:
            desired = -1.0  # short spread
        elif mt < lower[t]:
            desired = +1.0  # long spread

        if desired != pos:
            if params.enter_on_next_bar:
                pending_pos = desired
            else:
                pnl -= tc * (abs(desired - pos) * 2.0)
                if pos == 0.0 and desired != 0.0:
                    entries += 1
                if pos != 0.0 and desired == 0.0:
                    exits += 1
                pos = desired

    # force flat at end of day (optional)
    if pos != 0.0:
        pnl -= tc * (abs(0.0 - pos) * 2.0)
        exits += 1
        pos = 0.0

    return float(pnl), entries, exits

def trade_one_day_from_prev_pairs(
    trade_day: pl.Date,
    prev_pairs: pl.DataFrame,
    params: PairTradeParams
) -> tuple[float, int, int, int]:
    tickers = set(prev_pairs["leader"].to_list()) | set(prev_pairs["follower"].to_list())

    day_data: dict[str, pl.DataFrame] = {}
    base_ts = None
    for tk in tickers:
        try:
            df = load_ticker_day(tk, trade_day)
            if df.height == 0:
                continue
            day_data[tk] = df
            if base_ts is None:
                base_ts = df["timestamp"]
        except Exception:
            continue

    if not day_data or base_ts is None:
        return 0.0, 0, 0, 0

    ts_df = pl.DataFrame({"timestamp": base_ts})
    aligned: dict[str, np.ndarray] = {}
    for tk, df in day_data.items():
        df2 = ts_df.join(df, on="timestamp", how="left").with_columns(pl.col("r").fill_null(0.0))
        aligned[tk] = df2["r"].to_numpy()

    pair_pnls = []
    total_entries = 0
    total_exits = 0
    used = 0

    for row in prev_pairs.iter_rows(named=True):
        L = row["leader"]
        F = row["follower"]
        if (L not in aligned) or (F not in aligned):
            continue

        lag = int(round(abs(row["l_hat"])))
        if lag <= 0:
            continue

        pnl_pair, e, x = trade_one_pair_one_day_market_neutral(aligned[L], aligned[F], lag, params)
        # keep even if pnl=0; what we care about is trades now
        pair_pnls.append(pnl_pair)
        total_entries += e
        total_exits += x
        used += 1

    if used == 0:
        return 0.0, 0, 0, 0

    return float(np.mean(pair_pnls)), used, total_entries, total_exits

def run_backtest(params: PairTradeParams) -> pl.DataFrame:
    pairs = pl.read_parquet(PAIRS_PATH).sort("date")
    dates = pairs.select("date").unique().sort("date")["date"].to_list()

    rows = []
    total_entries = 0
    total_exits = 0

    for i in range(1, len(dates)):
        formation_day = dates[i - 1]
        trade_day     = dates[i]
        prev_pairs = pairs.filter(pl.col("date") == formation_day)

        pnl, n_used, e, x = trade_one_day_from_prev_pairs(trade_day, prev_pairs, params)
        total_entries += e
        total_exits += x
        rows.append((trade_day, pnl, n_used, e, x))

    out = pl.DataFrame(rows, schema=["date", "daily_pnl", "n_pairs_used", "entries", "exits"])
    print("TOTAL entries:", total_entries, "TOTAL exits:", total_exits)
    return out

params = PairTradeParams(
    lookback=20,   # like the other code
    k=2.0,
    z_exit=0.0,
    tc_bps=0.0,
    max_lag_minutes=30,
    enter_on_next_bar=True
)

bt = run_backtest(params)
print(bt.head(10))
print(bt.select([
    pl.len().alias("n_days"),
    pl.col("daily_pnl").mean().alias("mean_daily"),
    pl.col("daily_pnl").std().alias("std_daily"),
    pl.col("entries").sum().alias("sum_entries"),
    pl.col("exits").sum().alias("sum_exits"),
]))


TOTAL entries: 74769 TOTAL exits: 74769
shape: (10, 5)
┌────────────┬───────────┬──────────────┬─────────┬───────┐
│ date       ┆ daily_pnl ┆ n_pairs_used ┆ entries ┆ exits │
│ ---        ┆ ---       ┆ ---          ┆ ---     ┆ ---   │
│ date       ┆ f64       ┆ i64          ┆ i64     ┆ i64   │
╞════════════╪═══════════╪══════════════╪═════════╪═══════╡
│ 2015-01-05 ┆ -0.001545 ┆ 10           ┆ 150     ┆ 150   │
│ 2015-01-06 ┆ -0.001676 ┆ 10           ┆ 148     ┆ 148   │
│ 2015-01-07 ┆ -0.002121 ┆ 10           ┆ 154     ┆ 154   │
│ 2015-01-08 ┆ -0.002588 ┆ 10           ┆ 142     ┆ 142   │
│ 2015-01-09 ┆ -0.001113 ┆ 10           ┆ 145     ┆ 145   │
│ 2015-01-12 ┆ -0.000423 ┆ 10           ┆ 139     ┆ 139   │
│ 2015-01-13 ┆ 0.000319  ┆ 10           ┆ 157     ┆ 157   │
│ 2015-01-14 ┆ -0.004767 ┆ 10           ┆ 155     ┆ 155   │
│ 2015-01-15 ┆ -0.00613  ┆ 10           ┆ 141     ┆ 141   │
│ 2015-01-16 ┆ -0.00175  ┆ 10           ┆ 146     ┆ 146   │
└────────────┴───────────┴──────────────┴────

C:\Users\PC\AppData\Local\Temp\ipykernel_1876\2142423932.py:299: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  out = pl.DataFrame(rows, schema=["date", "daily_pnl", "n_pairs_used", "entries", "exits"])


## with hedging

### Trading strategy: 

Trading Logic: Based on the price difference m, the strategy generates entry and exit signals using Bollinger bands:

Entry Signal: If m[t] > upper[t], a short position is taken on the follower stock. If m[t] < lower[t], a long position is taken on the follower stock.

Exit Signal: If the price difference reverts to the mean (i.e., crosses the middle of the Bollinger bands), the position is closed.


- Long Position: When the price difference m[t] between the leader and follower is below the lower band, it indicates that the difference is significantly below average (so the follower stock is undervalued relative to the leader, according to the model). The expectation is that the price difference will revert upwards towards the mean (the follower stock will increase in value relative to the leader). So, we go long on the follower stock.
- Short Position: When the price difference m[t] is above the upper band, it suggests that the price difference is above average (so the follower stock is overvalued relative to the leader). The expectation is that the price difference will revert downwards, so we take a short position on the follower stock.

In [60]:
from __future__ import annotations

import math
from pathlib import Path
from dataclasses import dataclass
import numpy as np
import polars as pl

RETURNS_DIR = Path("data/selected/SP100/bbo")
PAIRS_PATH  = Path("data/top_pairs/daily_top_pairs_573_90.parquet")
SPY_TICKER = "SPY.P"

def load_ticker_day_df(ticker: str, day: pl.Date) -> pl.DataFrame:
    path = RETURNS_DIR / f"{ticker}.parquet"
    return (
        pl.scan_parquet(path)
        .filter(pl.col("timestamp").dt.date() == day)
        .select(
            pl.col("timestamp"),
            pl.col("mid_price_return").alias("r"),
        )
        .sort("timestamp")
        .collect()
    )

def align_to_grid(ts_df: pl.DataFrame, df: pl.DataFrame, col: str = "r") -> np.ndarray:
    # left join, fill missing with 0
    out = ts_df.join(df, on="timestamp", how="left").with_columns(pl.col(col).fill_null(0.0))
    return out[col].to_numpy()

def beta_to_spy(r_port: np.ndarray, r_spy: np.ndarray) -> float:
    r_port = np.where(np.isfinite(r_port), r_port, 0.0)
    r_spy  = np.where(np.isfinite(r_spy),  r_spy,  0.0)

    v = float(np.var(r_spy))
    if v < 1e-12:
        return 0.0

    cov = float(np.mean((r_port - r_port.mean()) * (r_spy - r_spy.mean())))
    return cov / v

import numpy as np
import polars as pl

import numpy as np
import polars as pl

def extract_trade_episodes(
    timestamps: np.ndarray,
    pos_series: np.ndarray,
    pnl_series: np.ndarray,
    min_hold_minutes: int = 5
) -> pl.DataFrame:
    """
    Convert minute-by-minute positions into trade episodes.

    Fix: timestamps are forced to int64 ns to avoid Polars Object dtype issues.
    """
    n = len(pos_series)
    if n == 0:
        return pl.DataFrame(
            schema={
                "entry_idx": pl.Int32,
                "exit_idx": pl.Int32,
                "entry_time": pl.Datetime("ns"),
                "exit_time": pl.Datetime("ns"),
                "direction": pl.Int8,
                "duration_min": pl.Int32,
                "trade_pnl": pl.Float64,
                "is_real": pl.Boolean,
            }
        )

    pos = np.asarray(pos_series, dtype=float)
    pnl = np.asarray(pnl_series, dtype=float)

    pos = np.where(np.isfinite(pos), pos, 0.0)
    pnl = np.where(np.isfinite(pnl), pnl, 0.0)

    # ---- key fix: make timestamps int64 nanoseconds ----
    ts = np.asarray(timestamps)
    if np.issubdtype(ts.dtype, np.datetime64):
        ts_ns = ts.astype("datetime64[ns]").astype(np.int64)
    else:
        # object array -> try converting
        ts_ns = np.array(ts, dtype="datetime64[ns]").astype(np.int64)

    episodes = []
    in_trade = False
    entry_idx = None
    direction = 0.0

    for t in range(n):
        pt = pos[t]
        if not in_trade:
            if pt != 0.0:
                in_trade = True
                entry_idx = t
                direction = pt
        else:
            if pt == 0.0:
                exit_idx = t
                trade_pnl = float(np.sum(pnl[entry_idx:exit_idx + 1]))
                duration = int(exit_idx - entry_idx + 1)
                episodes.append((entry_idx, exit_idx, direction, duration, trade_pnl))
                in_trade = False
                entry_idx = None
                direction = 0.0

    if in_trade and entry_idx is not None:
        exit_idx = n - 1
        trade_pnl = float(np.sum(pnl[entry_idx:exit_idx + 1]))
        duration = int(exit_idx - entry_idx + 1)
        episodes.append((entry_idx, exit_idx, direction, duration, trade_pnl))

    if len(episodes) == 0:
        return pl.DataFrame(
            {
                "entry_idx": [],
                "exit_idx": [],
                "entry_time": pl.Series([], dtype=pl.Datetime("ns")),
                "exit_time": pl.Series([], dtype=pl.Datetime("ns")),
                "direction": [],
                "duration_min": [],
                "trade_pnl": [],
                "is_real": [],
            }
        )

    entry_idx_list = [int(e[0]) for e in episodes]
    exit_idx_list  = [int(e[1]) for e in episodes]

    entry_time_ns = [int(ts_ns[i]) for i in entry_idx_list]
    exit_time_ns  = [int(ts_ns[i]) for i in exit_idx_list]

    df = pl.DataFrame(
        {
            "entry_idx": entry_idx_list,
            "exit_idx": exit_idx_list,
            "entry_time": pl.Series(entry_time_ns, dtype=pl.Datetime("ns")),
            "exit_time": pl.Series(exit_time_ns, dtype=pl.Datetime("ns")),
            "direction": [int(np.sign(e[2])) for e in episodes],
            "duration_min": [int(e[3]) for e in episodes],
            "trade_pnl": [float(e[4]) for e in episodes],
        }
    ).with_columns(
        (pl.col("duration_min") >= min_hold_minutes).alias("is_real")
    )

    return df



'''
def trade_one_pair_one_day_market_neutral(
    r_leader: np.ndarray,
    r_follower: np.ndarray,
    lag: int,
    params: PairTradeParams,
) -> tuple[np.ndarray, int, int]:
    """
    Returns: (pnl_series_per_minute, entries, exits)
    """
    n = min(len(r_leader), len(r_follower))
    if n == 0:
        return np.zeros(0), 0, 0

    lag = int(lag)
    if lag <= 0 or lag > params.max_lag_minutes:
        return np.zeros(n), 0, 0

    rL = np.where(np.isfinite(r_leader[:n]), r_leader[:n], 0.0)
    rF = np.where(np.isfinite(r_follower[:n]), r_follower[:n], 0.0)

    pL = np.cumsum(rL)
    pF = np.cumsum(rF)

    pred = np.full(n, np.nan, dtype=np.float64)
    pred[lag:] = pL[:-lag]
    m = pred - pF

    mu, sig = rolling_mean_std_nan_safe(m, params.lookback)
    sig[sig < 1e-12] = np.nan
    upper = mu + params.k * sig
    lower = mu - params.k * sig

    pos = 0.0
    entries = 0
    exits = 0
    tc = params.tc_bps / 10000.0

    start = max(params.lookback - 1, lag)
    pending_pos: float | None = None

    pnl = np.zeros(n, dtype=np.float64)

    for t in range(start, n):
        if params.enter_on_next_bar and pending_pos is not None:
            if pending_pos != pos:
                # 2 legs cost
                pnl[t] -= tc * (abs(pending_pos - pos) * 2.0)
                if pos == 0.0 and pending_pos != 0.0:
                    entries += 1
                if pos != 0.0 and pending_pos == 0.0:
                    exits += 1
                pos = pending_pos
            pending_pos = None

        # accrue pnl this minute
        pnl[t] += pos * rF[t] - pos * rL[t]

        mt = m[t]
        if not (np.isfinite(mt) and np.isfinite(mu[t]) and np.isfinite(upper[t]) and np.isfinite(lower[t])):
            continue

        # exit at mean
        if pos != 0.0:
            if (pos == +1.0 and mt >= mu[t] + params.z_exit) or (pos == -1.0 and mt <= mu[t] - params.z_exit):
                desired = 0.0
                if params.enter_on_next_bar:
                    pending_pos = desired
                else:
                    pnl[t] -= tc * (abs(desired - pos) * 2.0)
                    exits += 1
                    pos = desired
                continue

        # entry/flip on band breaks
        desired = pos
        if mt > upper[t]:
            desired = -1.0
        elif mt < lower[t]:
            desired = +1.0

        if desired != pos:
            if params.enter_on_next_bar:
                pending_pos = desired
            else:
                pnl[t] -= tc * (abs(desired - pos) * 2.0)
                if pos == 0.0 and desired != 0.0:
                    entries += 1
                if pos != 0.0 and desired == 0.0:
                    exits += 1
                pos = desired
    #to debug
    trades = extract_trade_episodes(
    timestamps=ts_array,   # numpy array of timestamps for that day/pair
    pos=pos,
    pnl=pnl,
    min_hold_minutes=5
    )
    
    print(trades.select([
        pl.len().alias("n_trades"),
        pl.col("is_real").sum().alias("n_real"),
        (pl.col("is_real").sum() / pl.len()).alias("frac_real"),
        pl.col("duration_min").median().alias("median_hold"),
        pl.col("trade_pnl").mean().alias("avg_trade_pnl"),
        pl.col("trade_pnl").sum().alias("sum_trade_pnl"),
    ]))
    #to debug

    # Force flat at end (charge cost on last bar so it appears in pnl[-1])
    if pos != 0.0:
        pnl[-1] -= tc * (abs(0.0 - pos) * 2.0)
        exits += 1
        pos = 0.0

    return pnl, entries, exits
'''

from dataclasses import dataclass
import numpy as np

@dataclass
class PairTradeParams:
    lookback: int = 20
    k: float = 2.0
    tc_bps: float = 0.0
    max_lag_minutes: int = 30
    enter_on_next_bar: bool = True

    # NEW: make trades “real”
    min_hold_minutes: int = 5
    target_return: float = 0.0   # e.g. 0.0005 means take profit at +5 bps (optional; 0 disables)
    allow_flip: bool = False     # paper-like behavior: usually False


import numpy as np
import polars as pl

def trade_one_pair_one_day_market_neutral(
    timestamps: np.ndarray,
    r_leader: np.ndarray,
    r_follower: np.ndarray,
    lag: int,
    params: PairTradeParams,
    debug: bool = False,
) -> tuple[np.ndarray, np.ndarray, int, int, pl.DataFrame]:
    """
    Returns:
      pnl_series (n,),
      pos_series (n,),
      entries,
      exits,
      trades_df (episodes)
    """
    n = min(len(r_leader), len(r_follower), len(timestamps))
    if n == 0:
        empty = np.zeros(0)
        return empty, empty, 0, 0, pl.DataFrame()

    lag = int(lag)
    if lag <= 0 or lag > params.max_lag_minutes:
        pnl = np.zeros(n)
        pos_series = np.zeros(n)
        trades = extract_trade_episodes(timestamps[:n], pos_series, pnl, params.min_hold_minutes)
        return pnl, pos_series, 0, 0, trades

    rL = np.where(np.isfinite(r_leader[:n]), r_leader[:n], 0.0)
    rF = np.where(np.isfinite(r_follower[:n]), r_follower[:n], 0.0)

    # price proxies
    pL = np.cumsum(rL)
    pF = np.cumsum(rF)

    pred = np.full(n, np.nan, dtype=float)
    pred[lag:] = pL[:-lag]
    m = pred - pF

    mu, sig = rolling_mean_std_nan_safe(m, params.lookback)
    sig = np.where(sig < 1e-12, np.nan, sig)

    upper = mu + params.k * sig
    lower = mu - params.k * sig

    tc = params.tc_bps / 10000.0

    pnl = np.zeros(n, dtype=float)
    pos_series = np.zeros(n, dtype=float)

    pos = 0.0
    pending_pos = None

    entries = 0
    exits = 0

    entry_idx = None
    cum_trade_pnl = 0.0

    start = max(params.lookback - 1, lag)

    for t in range(start, n):

        # apply delayed execution
        if params.enter_on_next_bar and pending_pos is not None:
            if pending_pos != pos:
                pnl[t] -= tc * (abs(pending_pos - pos) * 2.0)
                if pos == 0.0 and pending_pos != 0.0:
                    entries += 1
                    entry_idx = t
                    cum_trade_pnl = 0.0
                if pos != 0.0 and pending_pos == 0.0:
                    exits += 1
                    entry_idx = None
                    cum_trade_pnl = 0.0
                pos = pending_pos
            pending_pos = None

        # accrue pnl
        minute_pnl = pos * rF[t] - pos * rL[t]
        pnl[t] += minute_pnl
        pos_series[t] = pos

        if pos != 0.0 and entry_idx is not None:
            cum_trade_pnl += minute_pnl

        # indicators defined?
        if not (np.isfinite(m[t]) and np.isfinite(mu[t]) and np.isfinite(upper[t]) and np.isfinite(lower[t])):
            continue

        # ----- EXIT -----
        if pos != 0.0 and entry_idx is not None:
            held = t - entry_idx + 1
            if held >= params.min_hold_minutes:

                if params.target_return > 0.0 and cum_trade_pnl >= params.target_return:
                    desired = 0.0
                    pending_pos = desired if params.enter_on_next_bar else None
                    if not params.enter_on_next_bar:
                        pnl[t] -= tc * (abs(desired - pos) * 2.0)
                        exits += 1
                        pos = desired
                    continue

                # mean-cross exit
                if (pos == +1.0 and m[t] >= mu[t] + params.z_exit) or (pos == -1.0 and m[t] <= mu[t] - params.z_exit):
                    desired = 0.0
                    pending_pos = desired if params.enter_on_next_bar else None
                    if not params.enter_on_next_bar:
                        pnl[t] -= tc * (abs(desired - pos) * 2.0)
                        exits += 1
                        pos = desired
                    continue

        # ----- ENTRY (only when flat) -----
        if pos == 0.0:
            desired = 0.0
            if m[t] > upper[t]:
                desired = -1.0
            elif m[t] < lower[t]:
                desired = +1.0

            if desired != 0.0:
                if params.enter_on_next_bar:
                    pending_pos = desired
                else:
                    pnl[t] -= tc * (abs(desired - pos) * 2.0)
                    entries += 1
                    entry_idx = t
                    cum_trade_pnl = 0.0
                    pos = desired

        # flips disabled by default
        # (keep it off until you’re happy with “real trades”)

    # force flat end of day
    if pos != 0.0:
        pnl[-1] -= tc * (abs(0.0 - pos) * 2.0)
        exits += 1
        pos = 0.0
        pos_series[-1] = 0.0

    trades = extract_trade_episodes(
        timestamps=timestamps[:n],
        pos_series=pos_series,
        pnl_series=pnl,
        min_hold_minutes=params.min_hold_minutes
    )

    if debug:
        print(trades.select([
            pl.len().alias("n_trades"),
            pl.col("is_real").sum().alias("n_real"),
            (pl.col("is_real").sum() / pl.len()).alias("frac_real"),
            pl.col("duration_min").median().alias("median_hold"),
            pl.col("trade_pnl").mean().alias("avg_trade_pnl"),
            pl.col("trade_pnl").sum().alias("sum_trade_pnl"),
        ]))

    return pnl, pos_series, entries, exits, trades




'''
def trade_one_day_from_prev_pairs_with_spy_hedge(
    formation_day: pl.Date,
    trade_day: pl.Date,
    prev_pairs: pl.DataFrame,
    params: PairTradeParams
) -> tuple[float, int, int, int, float]:

    #to debug
    pos_series = np.zeros(n, dtype=np.float64)


    # ---------- TRADE day data ----------
    tickers = set(prev_pairs["leader"].to_list()) | set(prev_pairs["follower"].to_list())
    tickers.add(SPY_TICKER)

    day_data_trade: dict[str, pl.DataFrame] = {}
    base_ts = None

    for tk in tickers:
        try:
            df = load_ticker_day_df(tk, trade_day)
            if df.height == 0:
                continue
            day_data_trade[tk] = df
            if base_ts is None and tk != SPY_TICKER:
                base_ts = df["timestamp"]
        except Exception:
            continue

    if base_ts is None:
        return 0.0, 0, 0, 0, 0.0

    ts_df = pl.DataFrame({"timestamp": base_ts})
    n = ts_df.height

    # aligned trade day returns
    r_spy_trade = np.zeros(n, dtype=float)
    if SPY_TICKER in day_data_trade:
        r_spy_trade = align_to_grid(ts_df, day_data_trade[SPY_TICKER], col="r")

    aligned_trade: dict[str, np.ndarray] = {}
    for tk, df in day_data_trade.items():
        if tk == SPY_TICKER:
            continue
        aligned_trade[tk] = align_to_grid(ts_df, df, col="r")

    # ---------- Build portfolio minute pnl (unhedged) ----------
    pair_pnls = []
    total_entries = 0
    total_exits = 0
    used = 0

    for row in prev_pairs.iter_rows(named=True):
        L = row["leader"]
        F = row["follower"]
        if (L not in aligned_trade) or (F not in aligned_trade):
            continue

        lag = int(round(abs(row["l_hat"])))
        if lag <= 0:
            continue

        pnl_series, e, x = trade_one_pair_one_day_market_neutral(
            aligned_trade[L], aligned_trade[F], lag, params
        )
        if pnl_series.size != n:
            # safety (should match because we align)
            continue

        pair_pnls.append(pnl_series)
        total_entries += e
        total_exits += x
        used += 1

    if used == 0:
        return 0.0, 0, 0, 0, 0.0

    pnl_mat = np.vstack(pair_pnls)
    pnl_mat = np.nan_to_num(pnl_mat, nan=0.0)
    r_port_trade = pnl_mat.mean(axis=0)  # minute portfolio return

    # ---------- FORMATION day: compute beta ----------
    # Use the *same* weights on formation day: build r_port_formation by simulating the SAME pairs on formation day.
    # This avoids lookahead and is closest to “estimate hedge from formation sample”.

    # Load formation day returns for same tickers (+SPY)
    day_data_form: dict[str, pl.DataFrame] = {}
    base_ts_form = None

    for tk in tickers:
        try:
            df = load_ticker_day_df(tk, formation_day)
            if df.height == 0:
                continue
            day_data_form[tk] = df
            if base_ts_form is None and tk != SPY_TICKER:
                base_ts_form = df["timestamp"]
        except Exception:
            continue

    beta_used = 0.0
    if base_ts_form is not None and SPY_TICKER in day_data_form:
        ts_form = pl.DataFrame({"timestamp": base_ts_form})
        n_form = ts_form.height

        r_spy_form = align_to_grid(ts_form, day_data_form[SPY_TICKER], col="r")

        aligned_form: dict[str, np.ndarray] = {}
        for tk, df in day_data_form.items():
            if tk == SPY_TICKER:
                continue
            aligned_form[tk] = align_to_grid(ts_form, df, col="r")

        pair_form = []
        for row in prev_pairs.iter_rows(named=True):
            L = row["leader"]
            F = row["follower"]
            if (L not in aligned_form) or (F not in aligned_form):
                continue
            lag = int(round(abs(row["l_hat"])))
            if lag <= 0:
                continue
            pnl_series_form, _, _ = trade_one_pair_one_day_market_neutral(
                aligned_form[L], aligned_form[F], lag, params
            )
            if pnl_series_form.size != n_form:
                continue
            pair_form.append(pnl_series_form)

        if len(pair_form) > 0:
            r_port_form = np.vstack(pair_form).mean(axis=0)
            beta_used = beta_to_spy(r_port_form, r_spy_form)

    # ---------- Hedge on TRADE day ----------
    r_hedged = r_port_trade - beta_used * r_spy_trade
    daily_pnl_hedged = float(np.sum(r_hedged))

    return daily_pnl_hedged, used, total_entries, total_exits, float(beta_used)
'''

def trade_one_day_from_prev_pairs_with_spy_hedge(
    formation_day: pl.Date,
    trade_day: pl.Date,
    prev_pairs: pl.DataFrame,
    params: PairTradeParams,
    debug_one_pair: bool = False,
    debug_leader: str | None = None,
    debug_follower: str | None = None,
) -> tuple[float, int, int, int, float]:
    """
    Returns:
      daily_pnl_hedged, n_pairs_used, total_entries, total_exits, beta_used
    """

    # ---------- TRADE day data ----------
    tickers = set(prev_pairs["leader"].to_list()) | set(prev_pairs["follower"].to_list())
    tickers.add(SPY_TICKER)

    day_data_trade: dict[str, pl.DataFrame] = {}
    base_ts = None

    for tk in tickers:
        try:
            df = load_ticker_day_df(tk, trade_day)
            if df.height == 0:
                continue
            day_data_trade[tk] = df
            if base_ts is None and tk != SPY_TICKER:
                base_ts = df["timestamp"]
        except Exception:
            continue

    if base_ts is None:
        return 0.0, 0, 0, 0, 0.0

    ts_df = pl.DataFrame({"timestamp": base_ts})
    ts_array = ts_df["timestamp"].to_numpy()
    n = ts_df.height

    # align SPY on trade-day grid
    r_spy_trade = np.zeros(n, dtype=np.float64)
    if SPY_TICKER in day_data_trade:
        r_spy_trade = align_to_grid(ts_df, day_data_trade[SPY_TICKER], col="r")

    # align all stocks
    aligned_trade: dict[str, np.ndarray] = {}
    for tk, df in day_data_trade.items():
        if tk == SPY_TICKER:
            continue
        aligned_trade[tk] = align_to_grid(ts_df, df, col="r")

    # ---------- portfolio (unhedged) ----------
    pair_pnls = []
    total_entries = 0
    total_exits = 0
    used = 0

    for row in prev_pairs.iter_rows(named=True):
        L = row["leader"]
        F = row["follower"]
        if (L not in aligned_trade) or (F not in aligned_trade):
            continue

        lag = int(round(abs(float(row["l_hat"]))))
        if lag <= 0:
            continue

        debug = bool(
            debug_one_pair and debug_leader is not None and debug_follower is not None
            and (L == debug_leader and F == debug_follower)
        )

        pnl_series, pos_series, e, x, trades_df = trade_one_pair_one_day_market_neutral(
            timestamps=ts_array,
            r_leader=aligned_trade[L],
            r_follower=aligned_trade[F],
            lag=lag,
            params=params,
            debug=debug,
        )

        if pnl_series.size != n:
            continue

        pair_pnls.append(pnl_series)
        total_entries += e
        total_exits += x
        used += 1

    if used == 0:
        return 0.0, 0, 0, 0, 0.0

    pnl_mat = np.vstack(pair_pnls)
    pnl_mat = np.nan_to_num(pnl_mat, nan=0.0)
    r_port_trade = pnl_mat.mean(axis=0)

    # ---------- FORMATION day beta ----------
    day_data_form: dict[str, pl.DataFrame] = {}
    base_ts_form = None

    for tk in tickers:
        try:
            df = load_ticker_day_df(tk, formation_day)
            if df.height == 0:
                continue
            day_data_form[tk] = df
            if base_ts_form is None and tk != SPY_TICKER:
                base_ts_form = df["timestamp"]
        except Exception:
            continue

    beta_used = 0.0
    if base_ts_form is not None and SPY_TICKER in day_data_form:
        ts_form = pl.DataFrame({"timestamp": base_ts_form})
        ts_form_array = ts_form["timestamp"].to_numpy()
        n_form = ts_form.height

        r_spy_form = align_to_grid(ts_form, day_data_form[SPY_TICKER], col="r")

        aligned_form: dict[str, np.ndarray] = {}
        for tk, df in day_data_form.items():
            if tk == SPY_TICKER:
                continue
            aligned_form[tk] = align_to_grid(ts_form, df, col="r")

        pair_form = []
        for row in prev_pairs.iter_rows(named=True):
            L = row["leader"]
            F = row["follower"]
            if (L not in aligned_form) or (F not in aligned_form):
                continue

            lag = int(round(abs(float(row["l_hat"]))))
            if lag <= 0:
                continue

            pnl_series_form, _, _, _, _ = trade_one_pair_one_day_market_neutral(
                timestamps=ts_form_array,
                r_leader=aligned_form[L],
                r_follower=aligned_form[F],
                lag=lag,
                params=params,
                debug=False,
            )
            if pnl_series_form.size != n_form:
                continue

            pair_form.append(pnl_series_form)

        if len(pair_form) > 0:
            r_port_form = np.vstack(pair_form).mean(axis=0)
            beta_used = beta_to_spy(r_port_form, r_spy_form)

    # ---------- hedge ----------
    r_hedged = r_port_trade - beta_used * r_spy_trade
    daily_pnl_hedged = float(np.sum(r_hedged))

    return daily_pnl_hedged, used, total_entries, total_exits, float(beta_used)



def run_backtest(params: PairTradeParams) -> pl.DataFrame:
    pairs = pl.read_parquet(PAIRS_PATH).sort("date")
    dates = pairs.select("date").unique().sort("date")["date"].to_list()

    rows = []
    for i in range(1, len(dates)):
        formation_day = dates[i - 1]
        trade_day     = dates[i]
        prev_pairs = pairs.filter(pl.col("date") == formation_day)

        pnl, n_used, e, x, beta_used = trade_one_day_from_prev_pairs_with_spy_hedge(
            formation_day=formation_day,
            trade_day=trade_day,
            prev_pairs=prev_pairs,
            params=params
        )
        rows.append((trade_day, pnl, n_used, e, x, beta_used))

    return pl.DataFrame(rows, schema=["date", "daily_pnl_hedged", "n_pairs_used", "entries", "exits", "beta_spy"])

@dataclass
class PairTradeParams:
    lookback: int = 20
    k: float = 2.0
    z_exit: float = 0.0         
    tc_bps: float = 0.0
    max_lag_minutes: int = 30
    enter_on_next_bar: bool = True

    min_hold_minutes: int = 5
    target_return: float = 0.0
    allow_flip: bool = False



In [62]:
pairs = pl.read_parquet(PAIRS_PATH).sort("date")
dates = pairs.select("date").unique().sort("date")["date"].to_list()

formation_day = dates[0]
trade_day     = dates[1]
prev_pairs = pairs.filter(pl.col("date") == formation_day)

params = PairTradeParams(
    lookback=20,
    k=2.0,
    z_exit=0.0,
    tc_bps=0.0,
    max_lag_minutes=30,
    enter_on_next_bar=True,
    min_hold_minutes=5,     # <- real trades
    target_return=0.0,      # keep 0 for now
    allow_flip=False
)

pnl, used, e, x, beta = trade_one_day_from_prev_pairs_with_spy_hedge(
    formation_day=formation_day,
    trade_day=trade_day,
    prev_pairs=prev_pairs,
    params=params,
    debug_one_pair=True,
    debug_leader="AAPL.OQ",
    debug_follower="MSFT.OQ",
)

print("Daily hedged PnL:", pnl)
print("Pairs used:", used)
print("Entries:", e, "Exits:", x)
print("SPY beta used:", beta)


Daily hedged PnL: -0.001899114683038071
Pairs used: 10
Entries: 145 Exits: 145
SPY beta used: 0.004432368430882471


In [7]:
#entries are too high

In [52]:
pairs = pl.read_parquet(PAIRS_PATH).sort("date")

dates = pairs.select("date").unique().sort("date")["date"].to_list()

formation_day = dates[0]
trade_day     = dates[1]

prev_pairs = pairs.filter(pl.col("date") == formation_day)

from dataclasses import dataclass

@dataclass
class PairTradeParams:
    lookback: int = 20
    k: float = 2.0
    z_exit: float = 0.0           # <-- ADD THIS (used by exit logic)
    tc_bps: float = 0.0
    max_lag_minutes: int = 30
    enter_on_next_bar: bool = True

    # Real-trade controls
    min_hold_minutes: int = 5
    target_return: float = 0.0
    allow_flip: bool = False


pnl, used, e, x, beta = trade_one_day_from_prev_pairs_with_spy_hedge(
    formation_day=formation_day,
    trade_day=trade_day,
    prev_pairs=prev_pairs,
    params=params,
    debug_one_pair=True,
    debug_leader="AAPL.OQ",
    debug_follower="MSFT.OQ",
)

print("Daily hedged PnL:", pnl)
print("Pairs used:", used)
print("Entries:", e, "Exits:", x)
print("SPY beta used:", beta)


Daily hedged PnL: -0.0013922167601522059
Pairs used: 10
Entries: 150 Exits: 150
SPY beta used: 0.012670776050137542


In [ ]:
bt = run_backtest(params)
print(bt.head(10))
print(bt.select([
    pl.len().alias("n_days"),
    pl.col("daily_pnl_hedged").mean().alias("mean_daily"),
    pl.col("daily_pnl_hedged").std().alias("std_daily"),
    pl.col("entries").sum().alias("sum_entries"),
    pl.col("exits").sum().alias("sum_exits"),
    pl.col("beta_spy").mean().alias("mean_beta_spy"),
]))

In [46]:
import numpy as np
import polars as pl

def extract_trade_episodes(
    timestamps: np.ndarray,     # shape (n,) datetime64 or python datetimes
    pos: np.ndarray,            # shape (n,) values in {-1,0,+1}
    pnl: np.ndarray,            # shape (n,) per-minute pnl (already includes both legs)
    min_hold_minutes: int = 5,  # define "real"
) -> pl.DataFrame:
    """
    Returns one row per *round-trip* trade (entry->exit) with duration and PnL.

    If a trade is open at the end, it's ignored (or you can force-close before calling).
    """
    pos = np.asarray(pos, dtype=float)
    pnl = np.asarray(pnl, dtype=float)
    n = len(pos)
    assert len(pnl) == n
    assert len(timestamps) == n

    # changes in position
    prev = np.r_[0.0, pos[:-1]]
    dpos = pos - prev

    entry_idx = np.where((prev == 0.0) & (pos != 0.0))[0]
    exit_idx  = np.where((prev != 0.0) & (pos == 0.0))[0]

    trades = []
    j = 0  # pointer for exits

    for i in entry_idx:
        # find first exit after this entry
        while j < len(exit_idx) and exit_idx[j] <= i:
            j += 1
        if j >= len(exit_idx):
            break  # no exit found -> open trade at end

        x = exit_idx[j]
        direction = float(pos[i])  # +1 or -1
        duration = int(x - i)      # minutes held (approx)
        trade_pnl = float(np.nansum(pnl[i:x+1]))

        trades.append({
            "entry_i": i,
            "exit_i": x,
            "entry_ts": timestamps[i],
            "exit_ts": timestamps[x],
            "direction": direction,
            "duration_min": duration,
            "trade_pnl": trade_pnl,
            "is_real": duration >= min_hold_minutes,
        })

        j += 1

    return pl.DataFrame(trades) if trades else pl.DataFrame(
        schema={
            "entry_i": pl.Int64, "exit_i": pl.Int64,
            "entry_ts": pl.Datetime, "exit_ts": pl.Datetime,
            "direction": pl.Float64, "duration_min": pl.Int64,
            "trade_pnl": pl.Float64, "is_real": pl.Boolean,
        }
    )
